# Snapseed annotation

Hierarchical Snapseed labels for harmonized kidney organoid datasets. **Run the setup cell first**. Parameters live in `snapseed_annotation_config.yaml`.


In [ ]:
# import sys
# !{sys.executable} -m pip install -U scipy


In [ ]:
import random
import gc
import warnings
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import scanpy as sc
import snapseed
import yaml

CONFIG_PATH = Path("snapseed_annotation_config.yaml")

def load_annotation_config(config_path=CONFIG_PATH):
    with open(config_path) as fh:
        cfg = yaml.safe_load(fh)
    if cfg.get("hkoca_tools_root"):
        root = Path(cfg["hkoca_tools_root"]).expanduser().resolve()
    else:
        root = config_path.resolve().parent
        for _ in range(4):
            root = root.parent
    paths = {}
    for key, val in cfg["paths"].items():
        p = Path(val)
        paths[key] = p if p.is_absolute() else (root / val)
    return {"root": root, "paths": paths, "parameters": cfg.get("parameters", {})}

CFG = load_annotation_config()
P = CFG["parameters"]

BASE_DIR = CFG["paths"]["qc_input_dir"]
YAML_PATH = CFG["paths"]["marker_yaml"]
OUTPUT_DIR = CFG["paths"]["annotated_output_dir"]
CLUSTERED_DIR = CFG["paths"]["clustered_output_dir"]
SUMMARY_OUT = CFG["paths"]["summary_figures_dir"]
ADULT_REF_PATH = CFG["paths"]["adult_ref_h5ad"]
FETAL_REF_PATH = CFG["paths"]["fetal_ref_h5ad"]
ADULT_MARKERS_DIR = CFG["paths"]["adult_markers_dir"]
FETAL_MARKERS_DIR = CFG["paths"]["fetal_markers_dir"]
ANNOTATED_DIR = OUTPUT_DIR
INPUT_DIR = OUTPUT_DIR

SEED = P["seed"]
HVG_N_TOP_GENES = P["hvg_n_top_genes"]
PCA_N_COMPS = P["pca_n_comps"]
NEIGHBORS_N_NEIGHBORS = P["neighbors_n_neighbors"]
NEIGHBORS_N_PCS = P["neighbors_n_pcs"]
SCALE_MAX_VALUE = P["scale_max_value"]
NORMALIZE_TARGET_SUM = P["normalize_target_sum"]
FIGURE_DPI = P["dpi"]

plt.rcParams.update({
    "figure.dpi": FIGURE_DPI,
    "savefig.dpi": FIGURE_DPI,
})

random.seed(SEED)
np.random.seed(SEED)
sc.settings.seed = SEED
sc.settings.verbosity = P["scanpy_verbosity"]
anndata.settings.allow_write_nullable_strings = True

with open(YAML_PATH) as fh:
    MARKER_HIERARCHY = yaml.safe_load(fh)
print(f"HKOCA-tools root: {CFG['root']}")
print(f"Markers loaded from: {YAML_PATH}")
print(f"Top-level categories: {list(MARKER_HIERARCHY.keys())}")

def display_ranked_markers(adata, cluster_key, n_genes=15):
    """
    Top markers per cluster ranked by: logFC * (pct_in)^2 * (1 - pct_out)
    """
    if "rank_genes_groups" not in adata.uns:
        print("[-] Marker genes have not been ranked yet. Skipping display.")
        return
    groups = adata.uns["rank_genes_groups"]["names"].dtype.names
    print(f"\nDisplaying Top {n_genes} Marker Genes Per Cluster (Custom Score)")
    for cluster in groups:
        df = sc.get.rank_genes_groups_df(adata, group=cluster)
        df = df[df["logfoldchanges"] > 0].copy()  # Positive markers only
        df["score"] = (
            df["logfoldchanges"]
            * (df["pct_nz_group"] ** 2)
            * (1 - df["pct_nz_reference"])
        )
        df = df.sort_values("score", ascending=False).head(n_genes)
        df = df[[
            "names", "score", "logfoldchanges",
            "pct_nz_group", "pct_nz_reference", "pvals_adj"
        ]]
        df.columns = ["gene", "score", "logFC", "pct_in", "pct_out", "padj"]
        print(f"\nCluster {cluster} Top Markers:")
        display(df)

def _plot_umap_with_labels(adata, color_key, title="UMAP"):
    umap = adata.obsm["X_umap"]
    clusters = adata.obs[color_key].cat.categories
    cmap = plt.get_cmap("tab20", len(clusters))
    colors = {str(c): cmap(i) for i, c in enumerate(clusters)}
    fig, ax = plt.subplots(figsize=(8, 6), facecolor="#030303")
    ax.set_facecolor("#000000")
    for c in clusters:
        mask = adata.obs[color_key] == c
        ax.scatter(umap[mask, 0], umap[mask, 1], s=6, c=[colors[str(c)]], rasterized=True, alpha=0.8)
    for c in clusters:
        mask = adata.obs[color_key] == c
        if mask.sum() == 0:
            continue
        cx, cy = np.median(umap[mask, 0]), np.median(umap[mask, 1])
        ax.text(cx, cy, str(c), color="white", fontsize=10, fontweight="bold", ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.2", facecolor="black", alpha=0.7, edgecolor="white"))
    ax.set_title(title, color="white", fontsize=12, fontweight="bold")
    ax.set_xlabel("UMAP 1", color="#aaaaaa", fontsize=9)
    ax.set_ylabel("UMAP 2", color="#aaaaaa", fontsize=9)
    ax.tick_params(colors="#666666", labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.show()

def _plot_annotation_umaps_with_labels(adata, cols, title_prefix=""):
    umap = adata.obsm["X_umap"]
    n_panels = min(len(cols), 3)
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5.5), facecolor="#030303")
    if n_panels == 1:
        axes = [axes]
    cmaps = ["tab10", "tab20", "tab20b"]
    for ax, col, cmap_name in zip(axes, cols[:3], cmaps):
        ax.set_facecolor("#000000")
        try:
            cats = adata.obs[col].cat.categories
        except Exception:
            cats = sorted(adata.obs[col].dropna().unique())
        cmap = plt.get_cmap(cmap_name, max(len(cats), 1))
        palette = {cat: cmap(i) for i, cat in enumerate(cats)}
        ax.scatter(umap[:, 0], umap[:, 1], s=1, c="#1a1a1a", rasterized=True)
        for label, colour in palette.items():
            mask = adata.obs[col] == label
            if mask.sum() == 0:
                continue
            ax.scatter(umap[mask, 0], umap[mask, 1], s=4, c=[colour], rasterized=True, alpha=0.9)
            cx, cy = np.median(umap[mask, 0]), np.median(umap[mask, 1])
            ax.text(cx, cy, str(label), color="white", fontsize=8, fontweight="bold", ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7, edgecolor=colour, linewidth=1))
        ax.set_title(col, color="white", fontweight="bold", fontsize=11)
        ax.set_xlabel("UMAP 1", color="#aaaaaa", fontsize=8)
        ax.set_ylabel("UMAP 2", color="#aaaaaa", fontsize=8)
        ax.tick_params(colors="#666666", labelsize=8)
        for spine in ax.spines.values():
            spine.set_visible(False)
    plt.suptitle(f"Atlas Annotation Tiers: {title_prefix}", color="white", y=0.98, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


## Pipeline

`process_and_annotate_dataset` loads a QC-filtered object, clusters, runs Snapseed, and writes clustered + annotated h5ad files.


In [ ]:
def process_and_annotate_dataset(
    file_path,
    marker_dict,
    resolution=1.0,
    cluster_key="leiden_clusters",
    manual_annotations=None,
    output_dir=OUTPUT_DIR,
    clustered_dir=CLUSTERED_DIR,
):

    path = Path(file_path)
    print(f"\n{'='*70}\nPROCESSING: {path.name}\n{'='*70}")
    adata = sc.read_h5ad(path)
    adata.var_names = adata.var["features"].astype(str)
    adata.var_names_make_unique()
    print(f"[1/6] Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

    # Pre-scaled recovery protection check
    min_val = float(np.min(adata.X if isinstance(adata.X, np.ndarray) else adata.X.data))
    if min_val < 0:
        print("Warning: Detected negative values in adata.X! Assumed data is pre-scaled.")
        if adata.raw is not None:
            print("Restoring unscaled expression matrix from adata.raw...")
            adata = adata.raw.to_adata()
            adata.obsm.clear()
            adata.varm.clear()
        elif len(adata.layers) > 0:
            fallback_layer = next((l for l in ["counts", "raw", "normalized"] if l in adata.layers), None)
            if fallback_layer:
                print(f"Resetting main matrix to layer: '{fallback_layer}'")
                adata.X = adata.layers[fallback_layer].copy()
            else:
                raise ValueError("CRITICAL ERROR: adata.X is scaled, but raw/unscaled layer backups are missing.")
        else:
            raise ValueError("CRITICAL ERROR: Matrix is scaled with no fallback fields to recover unscaled counts.")

    max_val = float(np.max(adata.X if isinstance(adata.X, np.ndarray) else adata.X.data))
    if max_val > 100:
        print(f"[2/6] Raw Count Footprint Detected (Max expression = {max_val:.1f})")
        print("Normalizing total counts to 10,000...")
        sc.pp.normalize_total(adata, target_sum=NORMALIZE_TARGET_SUM)
        print("Computing log1p transform...")
        sc.pp.log1p(adata)
    else:
        print(f"[2/6] Input matrix appears pre-normalized (Max expression = {max_val:.1f}). Skipping Log-Norm.")
    print("[3/6] Selecting Highly Variable Genes...")
    sc.pp.highly_variable_genes(adata, n_top_genes=HVG_N_TOP_GENES, subset=False)
    print(f"Verified {adata.var['highly_variable'].sum():,} HVGs retained.")
    print("[4/6] Building isolated scaling matrix for PCA (Preserving pristine .X)...")
    adata_hvg = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.scale(adata_hvg, max_value=SCALE_MAX_VALUE)
    nan_mask = np.isnan(adata_hvg.X) if isinstance(adata_hvg.X, np.ndarray) else np.isnan(adata_hvg.X.data)

    if nan_mask.any():
        if isinstance(adata_hvg.X, np.ndarray): adata_hvg.X = np.nan_to_num(adata_hvg.X)
        else: adata_hvg.X.data = np.nan_to_num(adata_hvg.X.data)
        print("NaNs have been cleared and neutralized to 0.")
    print(f"[5/6] Computing Latent Dimension Spaces (PCA 50 -> Graph Neighbors k=20)...")
    sc.pp.pca(adata_hvg, n_comps=PCA_N_COMPS, svd_solver="arpack", random_state=SEED)
    adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
    del adata_hvg
    gc.collect()

    sc.pp.neighbors(adata, n_neighbors=NEIGHBORS_N_NEIGHBORS, n_pcs=NEIGHBORS_N_PCS)
    sc.tl.leiden(adata, resolution=resolution, key_added=cluster_key, random_state=SEED)
    sc.tl.umap(adata, random_state=SEED)
    adata.obs[cluster_key] = adata.obs[cluster_key].astype(str).astype("category")
    print(f"Clustering finished. Detected {adata.obs[cluster_key].nunique()} clusters.")

    # Clean reserved column names that break AnnData writes.
    if "_index" in adata.var.columns:
        adata.var.rename(columns={"_index": "var_index"}, inplace=True)
    if adata.raw is not None and "_index" in adata.raw.var.columns:
        adata.raw.var.rename(columns={"_index": "raw_index"}, inplace=True)

    # Save clustered object before annotation
    if clustered_dir is None:
        clustered_dir = CLUSTERED_DIR
    clustered_dir = Path(clustered_dir)
    clustered_dir.mkdir(parents=True, exist_ok=True)
    clustered_path = clustered_dir / f"{path.stem}_clustered.h5ad"
    adata.write_h5ad(clustered_path)
    print(f"Clustered object written -> {clustered_path}")

    _plot_umap_with_labels(adata, color_key=cluster_key, title=f"Pre-Annotation Base Clusters ({path.stem})")
    print("[6/6] Computing Differential Expression (Wilcoxon Rank-Sum)...")
    sc.tl.rank_genes_groups(adata, groupby=cluster_key, use_raw=False, method="wilcoxon", pts=True)
    print("Rendering Wilcoxon marker rank overview...")
    sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)
    print("Running Snapseed hierarchy annotation...")

    results = snapseed.annotate_hierarchy(adata, marker_dict, group_name=cluster_key)
    assignments = results["assignments"]
    metrics_df = results["metrics"]
    assignments.index = assignments.index.astype(str)
    assignments_col_lookup = {col.lower(): col for col in assignments.columns}

    def _resolve_assignment_col(level_key):
        if level_key in assignments.columns:
            return level_key
        key = str(level_key).strip().lower()
        if key in assignments_col_lookup:
            return assignments_col_lookup[key]
        if key.startswith("level_"):
            alt = "level" + key[len("level_"):]
            if alt in assignments_col_lookup:
                return assignments_col_lookup[alt]
        if key.startswith("level"):
            alt = key.replace("level", "level_")
            if alt in assignments_col_lookup:
                return assignments_col_lookup[alt]
        return None

    if manual_annotations is not None:
        if not isinstance(manual_annotations, dict):
            print("Warning: manual_annotations is not a dict. Skipping overrides.")
        else:
            print("Applying manual label overrides...")
            for cluster_id, levels in manual_annotations.items():
                cluster_id = str(cluster_id)
                if cluster_id not in assignments.index:
                    print(f"Warning: Cluster {cluster_id} not found in assignments index. Skipping.")
                    continue
                for level_name, value in levels.items():
                    resolved_col = _resolve_assignment_col(level_name)
                    if resolved_col is None:
                        print(f"Warning: Level '{level_name}' not found in assignments columns. Skipping.")
                        continue
                    assignments.loc[cluster_id, resolved_col] = value

    adata.uns["snapseed_assignments"] = assignments.fillna("Unassigned").astype(str)
    print("\nFinal Cluster Classification Assignments:")
    display(assignments)

    print("\nSnapseed Prediction Confidence Metrics:")
    if isinstance(metrics_df, pd.DataFrame):
        display(metrics_df)

    elif isinstance(metrics_df, dict):
        for lvl_key, lvl_metrics in metrics_df.items():
            print(f"Confidence Metrics Layer: {lvl_key}")
            display(lvl_metrics)

    adata.uns["snapseed_metrics"] = metrics_df
    level_labels = ["Level_1", "Level_2", "Level_3"]

    annotation_cols = []
    for i, col in enumerate(assignments.columns):
        obs_col = level_labels[i] if i < len(level_labels) else f"Level_{i + 1}"
        mapped = adata.obs[cluster_key].astype(str).map(assignments[col].to_dict())
        adata.obs[obs_col] = mapped.astype("category")
        if obs_col not in annotation_cols:
            annotation_cols.append(obs_col)

    latest = assignments.apply(
        lambda row: row.dropna().iloc[-1] if row.dropna().shape[0] > 0 else np.nan,
        axis=1,
    )

    fill = adata.obs[cluster_key].astype(str).map(latest.to_dict()).astype(object)
    for fallback in ["Level_2_fine", "Level_2", "Level_1_mid", "Level_1"]:
        if fallback in adata.obs:
            fill = fill.fillna(adata.obs[fallback].astype(object))

    fill = fill.fillna(adata.obs[cluster_key].astype(str))
    adata.obs["Level_3_latest"] = fill.astype("category")

    if "Level_3_latest" not in annotation_cols:
        annotation_cols.append("Level_3_latest")
    _plot_annotation_umaps_with_labels(
        adata,
        annotation_cols[:2] + ["Level_3_latest"],
        title_prefix=path.stem,
    )

    if output_dir is None:
        output_dir = OUTPUT_DIR
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{path.stem}_annotated.h5ad"
    adata.write_h5ad(out_path)
    
    print(f"\nProcess Complete. File Written -> {out_path}\n" + "-"*70)
    return adata


## Per-dataset annotation

One cell per study. Tune `chosen_resolution` and `curated_overrides` as needed. Optional plots at the bottom of each cell are commented out.


In [ ]:
dataset_name = "d10_1016_j_cell_2020_04_004_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_cell_2020_04_004
chosen_resolution = 0.6
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "6": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    }
}
adata_1016_j_cell_2020_04_004 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_cell_2020_04_004, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_cell_2020_04_004, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_cell_2020_04_004, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1016_j_celrep_2020_108514_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_celrep_2020_108514
chosen_resolution = 0.4
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "5": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    },
    "9": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
}
}
adata_1016_j_celrep_2020_108514 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_celrep_2020_108514, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_celrep_2020_108514, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_celrep_2020_108514, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


**QC note:** Run the first pass, inspect the violin plot, drop the flagged cluster, then re-annotate. Keep the existing neighbors/UMAP.


In [ ]:
dataset_name = "d10_1016_j_cmet_2022_04_009_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_cmet_2022_04_009
chosen_resolution = 0.3
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "5": {
        "Level_1": "Off-target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    }
}
adata_1016_j_cmet_2022_04_009 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
sc.pl.violin(adata_1016_j_cmet_2022_04_009, keys=["nCount_RNA", "nFeature_RNA"], groupby=dynamic_cluster_key)

# QC outlier cluster; keep neighbors/UMAP from initial clustering
bad_cluster = "3"
adata_1016_j_cmet_2022_04_009 = adata_1016_j_cmet_2022_04_009[~adata_1016_j_cmet_2022_04_009.obs[dynamic_cluster_key].isin([bad_cluster])].copy()

# Annotation on the cleaned data (uses existing neighbors/UMAP)
print("\n--- Running annotation on cleaned data ---")
results = snapseed.annotate_hierarchy(adata_1016_j_cmet_2022_04_009, MARKER_HIERARCHY, group_name=dynamic_cluster_key)
assignments = results["assignments"]
metrics_df = results["metrics"]
print("\nCluster Assignments:")
display(assignments)
level_labels = ["Level_1", "Level_2", "Level_3"]
assignments_str = assignments.copy()
assignments_str.index = assignments_str.index.astype(str)
for i, col in enumerate(assignments.columns):
    obs_col = level_labels[i] if i < len(level_labels) else f"Level_{i}_annotation"
    adata_1016_j_cmet_2022_04_009.obs[obs_col] = (
        adata_1016_j_cmet_2022_04_009.obs[dynamic_cluster_key]
        .map(assignments_str[col].to_dict())
        .astype("category")
    )
latest = assignments_str.apply(
    lambda row: row.dropna().iloc[-1] if row.dropna().shape[0] > 0 else np.nan,
    axis=1,
)
fill = adata_1016_j_cmet_2022_04_009.obs[dynamic_cluster_key].astype(str).map(latest.to_dict()).astype(object)  # required for manual label edits on categorical obs
for fallback in ["Level_2", "Level_1"]:
    if fallback in adata_1016_j_cmet_2022_04_009.obs:
        fill = fill.fillna(adata_1016_j_cmet_2022_04_009.obs[fallback].astype(object))
fill = fill.fillna(adata_1016_j_cmet_2022_04_009.obs[dynamic_cluster_key].astype(str))
adata_1016_j_cmet_2022_04_009.obs["Level_3_latest"] = fill.astype("category")
_plot_annotation_umaps_with_labels(
    adata_1016_j_cmet_2022_04_009, ["Level_1", "Level_2", "Level_3_latest"], title_prefix=dataset_name
)

# display_ranked_markers(adata_1016_j_cmet_2022_04_009, cluster_key=dynamic_cluster_key, n_genes=15)
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / f"{Path(dataset_name).stem}_cleaned_annotated.h5ad"
adata_1016_j_cmet_2022_04_009.write_h5ad(out_path)
print(f"Saved -> {out_path}")


In [ ]:
dataset_name = "d10_1016_j_devcel_2019_06_001_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_devcel_2019_06_001
chosen_resolution = 0.3
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "7": {
        "Level_1": "Off_target_cells",
        "Level_2": "Muscle_progenitor_cells",
        "Level_3": "Muscle_progenitor_cells"
    },
    "8": {
        "Level_1": "Nephron",
        "Level_2": "Proliferative_tubule",
        "Level_3": "Proliferative_tubule"
    },
    "1": {
        "Level_1": "Stromal_cells",
        "Level_2": "Activated Fibroblasts",
        "Level_3": "Activated Fibroblasts"
    },
    "4": {
        "Level_1": "Nephron",
        "Level_2": "Proximal_tubule",
        "Level_3": "PT"
    },
    "10": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "3": {
        "Level_1": "Stromal_cells",
        "Level_2": "Interstitial_Fibroblasts",
        "Level_3": "Interstitial_Fibroblasts"
    }
}
adata_1016_j_devcel_2019_06_001 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides,
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_devcel_2019_06_001, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_devcel_2019_06_001, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_devcel_2019_06_001, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1016_j_stem_2018_04_022_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_stem_2018_04_022
chosen_resolution = 0.6
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = None
{
    "6": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    }
}
adata_1016_j_stem_2018_04_022 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_stem_2018_04_022, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_stem_2018_04_022, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_stem_2018_04_022, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1016_j_stem_2018_10_010_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_stem_2018_10_010
chosen_resolution = 0.3
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "11": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "9": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "2": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "6": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
}
adata_1016_j_stem_2018_10_010 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_stem_2018_10_010, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_stem_2018_10_010, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_stem_2018_10_010, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1016_j_stem_2019_06_009_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
chosen_resolution = 0.5
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "10": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "4": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "7": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "12": {
        "Level_1": "Stromal_cells",
        "Level_2": "Interstitial_Fibroblasts",
        "Level_3": "Interstitial_Fibroblasts"
    },
    "8": {
        "Level_1": "Stromal_cells",
        "Level_2": "Mesangial_cells",
        "Level_3": "Mesangial_cells"
    },
    "3": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    },
    "5": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    }
}
print("\n--- Running initial core computation ---")
adata_1016_j_stem_2019_06_009 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides  # Overrides applied AFTER regression
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_stem_2019_06_009, cluster_key=dynamic_cluster_key, n_genes=15)


In [ ]:
dataset_name = "d10_1016_j_stem_2021_11_001_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_stem_2021_11_001
chosen_resolution = 0.4
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = None
"""
{
 "17": {
 "Level_1": "Off_target_cells",
 "Level_2": "Off_target_cells",
 "Level_3": "Off_target_cells"
 }
}
"""
adata_1016_j_stem_2021_11_001 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_stem_2021_11_001, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1016_j_stem_2021_11_001, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1016_j_stem_2021_11_001, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


**QC note:** Cluster first without overrides, remove the outlier cluster, then apply `curated_overrides` manually.


In [ ]:
dataset_name = "d10_1016_j_stem_2021_12_010_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
chosen_resolution = 0.3
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
show_markers_tables = True
curated_overrides = {
    "1": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "8": {
        "Level_1": "Nephron",
        "Level_2": "Podocytes",
        "Level_3": "Mature_podocytes"
    }
}
print("\n--- Running initial core computation ---")
adata_1016_j_stem_2021_12_010 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations={}
)
print("\n--- Plotting initial annotations ---")
_plot_annotation_umaps_with_labels(
    adata_1016_j_stem_2021_12_010,
    ["Level_1", "Level_2", "Level_3"],
    title_prefix=f"{dataset_name} | Initial Run (Res {chosen_resolution})"
)
print("\n--- Plotting QC Violins ---")
sc.pl.violin(
    adata_1016_j_stem_2021_12_010,
    keys=["nCount_RNA", "nFeature_RNA"],
    groupby=dynamic_cluster_key
)
print("\n--- Removing Bad Cluster (1) ---")

# QC outlier cluster; keep neighbors/UMAP from initial clustering
bad_cluster = "0"
adata_1016_j_stem_2021_12_010 = adata_1016_j_stem_2021_12_010[
    ~adata_1016_j_stem_2021_12_010.obs[dynamic_cluster_key].isin([bad_cluster])
].copy()
print("\n--- Applying Curated Overrides ---")
for lvl in ["Level_1", "Level_2", "Level_3"]:
    if lvl in adata_1016_j_stem_2021_12_010.obs.columns:
        adata_1016_j_stem_2021_12_010.obs[lvl] = adata_1016_j_stem_2021_12_010.obs[lvl].astype(object)
for cluster_id, annotations in curated_overrides.items():
    mask = adata_1016_j_stem_2021_12_010.obs[dynamic_cluster_key] == cluster_id
    if mask.any():
        if "Level_1" in annotations: adata_1016_j_stem_2021_12_010.obs.loc[mask, "Level_1"] = annotations["Level_1"]
        if "Level_2" in annotations: adata_1016_j_stem_2021_12_010.obs.loc[mask, "Level_2"] = annotations["Level_2"]
        if "Level_3" in annotations: adata_1016_j_stem_2021_12_010.obs.loc[mask, "Level_3"] = annotations["Level_3"]
print("\n--- Finalizing Level Fallbacks (Level 3 -> Level 2) ---")
fill = adata_1016_j_stem_2021_12_010.obs["Level_3"]
if "Level_2" in adata_1016_j_stem_2021_12_010.obs.columns:
    fill = fill.fillna(adata_1016_j_stem_2021_12_010.obs["Level_2"])
if "Level_1" in adata_1016_j_stem_2021_12_010.obs.columns:
    fill = fill.fillna(adata_1016_j_stem_2021_12_010.obs["Level_1"])
fill = fill.fillna(adata_1016_j_stem_2021_12_010.obs[dynamic_cluster_key].astype(str))
adata_1016_j_stem_2021_12_010.obs["Level_3_latest"] = fill.astype("category")
for lvl in ["Level_1", "Level_2", "Level_3"]:
    if lvl in adata_1016_j_stem_2021_12_010.obs.columns:
        adata_1016_j_stem_2021_12_010.obs[lvl] = adata_1016_j_stem_2021_12_010.obs[lvl].astype("category")
print("\n--- Final Annotation UMAPs (Post-QC & Post-Overrides) ---")
_plot_annotation_umaps_with_labels(
    adata_1016_j_stem_2021_12_010,
    ["Level_1", "Level_2", "Level_3_latest"],
    title_prefix=f"{dataset_name} | FINAL (Res {chosen_resolution}, clst {bad_cluster} removed)"
)
print("\n--- Saving Final Cleaned AnnData ---")
cleaned_filename = dataset_name.replace(".h5ad", "_final_cleaned.h5ad")
final_output_path = str(OUTPUT_DIR / cleaned_filename)
adata_1016_j_stem_2021_12_010.write_h5ad(final_output_path)
print(f"Saved successfully to: {final_output_path}")
if show_markers_tables:
    display_ranked_markers(
        adata_1016_j_stem_2021_12_010,
        cluster_key=dynamic_cluster_key,
        n_genes=15
    )
sc.pl.umap(adata_1016_j_stem_2021_12_010, color=["HES6", "STMN2", "MYOG", "FABP7", "SRGN", "PTPRC", "OTX2", "CDX2", "TUBB3", "MYOD1", "MYL1"], cmap="viridis", use_raw=False)
sc.pl.umap(adata_1016_j_stem_2021_12_010, color=["COL14A1", "COL1A1", "COL2A1", "PDGFRA", "PDGFRB", "MEIS1", "MEIS2"], cmap="viridis", use_raw=False)
sc.pl.umap(adata_1016_j_stem_2021_12_010, color=["NPHS1", "NPHS2", "EPCAM", "CLDN1", "EYA1", "PAX2", "CUBN", "CALB1", "SIX1", "SIX2", "EYA1", "PAX2", "CITED1", "PAX8", "LYPD1"], cmap="viridis", use_raw=False)


**QC note:** Subset the bad cluster, re-run Snapseed, and keep manual overrides (cast labels to object before editing).


In [ ]:
dataset_name = "d10_1016_j_stem_2022_06_005_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1016_j_stem_2022_06_005
chosen_resolution = 0.1
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "1": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "0": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "3": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    }
}
adata_1016_j_stem_2022_06_005 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
sc.pl.violin(adata_1016_j_stem_2022_06_005, keys=["nCount_RNA", "nFeature_RNA"], groupby=dynamic_cluster_key)

# QC outlier cluster; keep neighbors/UMAP from initial clustering
bad_cluster = "10"
print(f"\n--- Removing bad cluster: {bad_cluster} ---")
print(f"Cells before removal: {adata_1016_j_stem_2022_06_005.n_obs}")
remaining_clusters = [c for c in adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].unique()
                      if c != bad_cluster]
print(f"Remaining clusters: {remaining_clusters}")
adata_1016_j_stem_2022_06_005 = adata_1016_j_stem_2022_06_005[
    ~adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].isin([bad_cluster])
].copy()
print(f"Cells after removal: {adata_1016_j_stem_2022_06_005.n_obs}")
print("\n--- Running annotation on cleaned data ---")
results = snapseed.annotate_hierarchy(
    adata_1016_j_stem_2022_06_005,
    MARKER_HIERARCHY,
    group_name=dynamic_cluster_key
)
assignments = results["assignments"]
metrics_df = results["metrics"]
print("\nCluster Assignments:")
display(assignments)
level_labels = ["Level_1", "Level_2", "Level_3"]
assignments_str = assignments.copy()
assignments_str.index = assignments_str.index.astype(str)
for i, col in enumerate(assignments.columns):
    obs_col = level_labels[i] if i < len(level_labels) else f"Level_{i}_annotation"
    adata_1016_j_stem_2022_06_005.obs[obs_col] = (
        adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key]
        .map(assignments_str[col].to_dict())
        .astype(object)   # required for manual label edits on categorical obs
    )
print("\n--- Re-applying manual overrides ---")
for cluster_id, annotations in curated_overrides.items():
    if cluster_id in remaining_clusters:  # Only if this cluster still exists
        for level, annotation in annotations.items():
            if level in adata_1016_j_stem_2022_06_005.obs.columns:
                mask = adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].astype(str) == cluster_id
                adata_1016_j_stem_2022_06_005.obs.loc[mask, level] = annotation
                print(f"Applied manual annotation: Cluster {cluster_id} -> {level} = '{annotation}'")
latest = assignments_str.apply(
    lambda row: row.dropna().iloc[-1] if row.dropna().shape[0] > 0 else np.nan,
    axis=1,
)
fill = adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].astype(str).map(latest.to_dict()).astype(object)
for cluster_id, annotations in curated_overrides.items():
    if cluster_id in remaining_clusters and "Level_3" in annotations:
        mask = adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].astype(str) == cluster_id
        fill[mask] = annotations["Level_3"]
for fallback in ["Level_2", "Level_1"]:
    if fallback in adata_1016_j_stem_2022_06_005.obs:
        fill = fill.fillna(adata_1016_j_stem_2022_06_005.obs[fallback].astype(object))
fill = fill.fillna(adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].astype(str))
adata_1016_j_stem_2022_06_005.obs["Level_3_latest"] = fill.astype(object)
print("\n--- Re-categorizing columns ---")
for lvl in ["Level_1", "Level_2", "Level_3", "Level_3_latest"]:
    if lvl in adata_1016_j_stem_2022_06_005.obs.columns:
        adata_1016_j_stem_2022_06_005.obs[lvl] = adata_1016_j_stem_2022_06_005.obs[lvl].astype("category")
print("\n--- Verifying manual overrides in final object ---")
for cluster_id in curated_overrides.keys():
    if cluster_id in remaining_clusters:
        cluster_cells = adata_1016_j_stem_2022_06_005.obs[
            adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].astype(str) == cluster_id
        ]
        if len(cluster_cells) > 0:
            print(f"Cluster {cluster_id}:")
            print(f"Level_1: {cluster_cells['Level_1'].iloc[0]}")
            print(f"Level_2: {cluster_cells['Level_2'].iloc[0]}")
            print(f"Level_3: {cluster_cells['Level_3'].iloc[0]}")
            print(f"Level_3_latest: {cluster_cells['Level_3_latest'].iloc[0]}")
_plot_annotation_umaps_with_labels(
    adata_1016_j_stem_2022_06_005,
    ["Level_1", "Level_2", "Level_3_latest"],
    title_prefix=f"{dataset_name}_cleaned"
)
if show_markers_tables:
    display_ranked_markers(adata_1016_j_stem_2022_06_005, cluster_key=dynamic_cluster_key, n_genes=15)
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / f"{Path(dataset_name).stem}_cleaned_annotated.h5ad"
adata_1016_j_stem_2022_06_005.write_h5ad(out_path)
print(f"\nSaved final annotated object -> {out_path}")
print("\n--- Final Annotation Summary ---")
print(f"Total cells: {adata_1016_j_stem_2022_06_005.n_obs}")
print(f"Remaining clusters: {sorted(adata_1016_j_stem_2022_06_005.obs[dynamic_cluster_key].unique())}")
print("\nLevel_1 distribution:")
display(adata_1016_j_stem_2022_06_005.obs['Level_1'].value_counts())
print("\nLevel_2 distribution:")
display(adata_1016_j_stem_2022_06_005.obs['Level_2'].value_counts())
print("\nLevel_3_latest distribution:")
display(adata_1016_j_stem_2022_06_005.obs['Level_3_latest'].value_counts())


**QC note:** Outlier cluster removal followed by re-annotation on the trimmed object.


In [ ]:
dataset_name = "d10_1038_s41467_019_13382_0_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1038_s41467_019_13382_0
chosen_resolution = 0.3
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "4": {
        "Level_1": "Stromal_cells",
        "Level_2": "Mesangial_cells",
        "Level_3": "Mesangial_cells"
    },
    "7": {
        "Level_1": "Off_target_cells",
        "Level_2": "Muscle_progenitor_cells",
        "Level_3": "Muscle_progenitor_cells"
    },
    "8": {
        "Level_1": "Off_target_cells",
        "Level_2": "undifferentiated_pluripotent_cells",
        "Level_3": "undifferentiated_pluripotent_cells"
    },
    "2": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    }
}
adata_1038_s41467_019_13382_0 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
sc.pl.violin(adata_1038_s41467_019_13382_0, keys=["nCount_RNA", "nFeature_RNA"], groupby=dynamic_cluster_key)

# QC outlier cluster; keep neighbors/UMAP from initial clustering
bad_cluster = "11"
print(f"\n--- Removing bad cluster: {bad_cluster} ---")
print(f"Cells before removal: {adata_1038_s41467_019_13382_0.n_obs}")
remaining_clusters = [c for c in adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].unique()
                      if c != bad_cluster]
print(f"Remaining clusters: {remaining_clusters}")
adata_1038_s41467_019_13382_0 = adata_1038_s41467_019_13382_0[
    ~adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].isin([bad_cluster])
].copy()
print(f"Cells after removal: {adata_1038_s41467_019_13382_0.n_obs}")
print("\n--- Running annotation on cleaned data ---")
results = snapseed.annotate_hierarchy(
    adata_1038_s41467_019_13382_0,
    MARKER_HIERARCHY,
    group_name=dynamic_cluster_key
)
assignments = results["assignments"]
metrics_df = results["metrics"]
print("\nCluster Assignments:")
display(assignments)
level_labels = ["Level_1", "Level_2", "Level_3"]
assignments_str = assignments.copy()
assignments_str.index = assignments_str.index.astype(str)
for i, col in enumerate(assignments.columns):
    obs_col = level_labels[i] if i < len(level_labels) else f"Level_{i}_annotation"
    adata_1038_s41467_019_13382_0.obs[obs_col] = (
        adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key]
        .map(assignments_str[col].to_dict())
        .astype(object)   # required for manual label edits on categorical obs
    )
print("\n--- Re-applying manual overrides ---")
for cluster_id, annotations in curated_overrides.items():
    if cluster_id in remaining_clusters:  # Only if this cluster still exists
        for level, annotation in annotations.items():
            if level in adata_1038_s41467_019_13382_0.obs.columns:
                mask = adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].astype(str) == cluster_id
                adata_1038_s41467_019_13382_0.obs.loc[mask, level] = annotation
                print(f"Applied manual annotation: Cluster {cluster_id} -> {level} = '{annotation}'")
latest = assignments_str.apply(
    lambda row: row.dropna().iloc[-1] if row.dropna().shape[0] > 0 else np.nan,
    axis=1,
)
fill = adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].astype(str).map(latest.to_dict()).astype(object)
for cluster_id, annotations in curated_overrides.items():
    if cluster_id in remaining_clusters and "Level_3" in annotations:
        mask = adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].astype(str) == cluster_id
        fill[mask] = annotations["Level_3"]
for fallback in ["Level_2", "Level_1"]:
    if fallback in adata_1038_s41467_019_13382_0.obs:
        fill = fill.fillna(adata_1038_s41467_019_13382_0.obs[fallback].astype(object))
fill = fill.fillna(adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].astype(str))
adata_1038_s41467_019_13382_0.obs["Level_3_latest"] = fill.astype(object)
print("\n--- Re-categorizing columns ---")
for lvl in ["Level_1", "Level_2", "Level_3", "Level_3_latest"]:
    if lvl in adata_1038_s41467_019_13382_0.obs.columns:
        adata_1038_s41467_019_13382_0.obs[lvl] = adata_1038_s41467_019_13382_0.obs[lvl].astype("category")
print("\n--- Verifying manual overrides in final object ---")
for cluster_id in curated_overrides.keys():
    if cluster_id in remaining_clusters:
        cluster_cells = adata_1038_s41467_019_13382_0.obs[
            adata_1038_s41467_019_13382_0.obs[dynamic_cluster_key].astype(str) == cluster_id
        ]
        if len(cluster_cells) > 0:
            print(f"Cluster {cluster_id}:")
            print(f"Level_1: {cluster_cells['Level_1'].iloc[0]}")
            print(f"Level_2: {cluster_cells['Level_2'].iloc[0]}")
            print(f"Level_3: {cluster_cells['Level_3'].iloc[0]}")
            print(f"Level_3_latest: {cluster_cells['Level_3_latest'].iloc[0]}")
_plot_annotation_umaps_with_labels(
    adata_1038_s41467_019_13382_0,
    ["Level_1", "Level_2", "Level_3_latest"],
    title_prefix=f"{dataset_name}_cleaned"
)
if show_markers_tables:
    display_ranked_markers(adata_1038_s41467_019_13382_0, cluster_key=dynamic_cluster_key, n_genes=15)
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / f"{Path(dataset_name).stem}_cleaned_annotated.h5ad"
adata_1038_s41467_019_13382_0.write_h5ad(out_path)
print(f"\nSaved final annotated object -> {out_path}")


In [ ]:
dataset_name = "d10_1038_s41467_022_33623_z_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_d10_1038_s41467_022_33623_z
chosen_resolution = 0.9
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "12": {
        "Level_1": "Stromal_cells",
        "Level_2": "Activated Fibroblasts",
        "Level_3": "Activated Fibroblasts"
    },
    "15": {
        "Level_1": "Stromal_cells",
        "Level_2": "Activated Fibroblasts",
        "Level_3": "Activated Fibroblasts"
    }
}
adata_d10_1038_s41467_022_33623_z = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_d10_1038_s41467_022_33623_z, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(adata_d10_1038_s41467_022_33623_z, color=["NPHS1", "NPHS2", "EPCAM", "CLDN1", "EYA1", "PAX2", "CUBN", "CALB1", "ACTA2", "POSTN"], cmap="viridis", use_raw=False)

# sc.pl.dotplot(adata_d10_1038_s41467_022_33623_z, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1038_s41551_025_01542_1_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1038_s41551_025_01542_1
chosen_resolution = 0.4
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "7": {
        "Level_1": "Stromal_cells",
        "Level_2": "Stromal_progenitors",
        "Level_3": "Stromal_progenitors"
    },
    "10": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "8": {
        "Level_1": "Stromal_cells",
        "Level_2": "Pericytes",
        "Level_3": "Pericytes"
    }
}
adata_1038_s41551_025_01542_1 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1038_s41551_025_01542_1, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1038_s41551_025_01542_1, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1038_s41551_025_01542_1, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1038_s41563_020_00853_9_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1038_s41563_020_00853_9
chosen_resolution = 0.8
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "14": {
        "Level_1": "Endothelial_cells",
        "Level_2": "Early_endothelium",
        "Level_3": "Early_endothelium"
    },
    "10": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    }
}
adata_1038_s41563_020_00853_9 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1038_s41563_020_00853_9, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(adata_1038_s41563_020_00853_9, color=["COL14A1", "COL1A1", "PDGFRA", "PDGFRB", "MEIS1", "MEIS2", "UBE2C", "COL6A3", "COL14A1", "COL5A1", "COL11A1", "COL12A1", "C7", "COL3A1", "COL1A1", "DCN", "PDGFRA", "ACTA2", "POSTN"], cmap="viridis", use_raw=False)

# sc.pl.dotplot(adata_1038_s41563_020_00853_9, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1038_s41592_018_0253_2_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1038_s41592_018_0253_2
chosen_resolution = 0.5
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = None
"""
{
 "17": {
 "Level_1": "Off_target_cells",
 "Level_2": "Off_target_cells",
 "Level_3": "Off_target_cells"
 }
}
"""
adata_1038_s41592_018_0253_2 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1038_s41592_018_0253_2, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1038_s41592_018_0253_2, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1038_s41592_018_0253_2, var_names=["CDX2", "FOXA1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1038_s42003_024_07069_6_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1038_s42003_024_07069_6
chosen_resolution = 0.4
show_markers_tables = False
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "10": {
        "Level_1": "Off_target_cells",
        "Level_2": "Muscle_progenitor_cells",
        "Level_3": "Muscle_progenitor_cells"
    },
    "13": {
        "Level_1": "Nephron",
        "Level_2": "Proliferative_tubule",
        "Level_3": "Proliferative_tubule"
    },
    "8": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    }
}
adata_1038_s42003_024_07069_6 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1038_s42003_024_07069_6, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1038_s42003_024_07069_6, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1038_s42003_024_07069_6, var_names=["CDX2", "FOXA1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1101_505396_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1101_505396
chosen_resolution = 0.6
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "8": {
        "Level_1": "Nephron",
        "Level_2": "Nephron_progenitor_cells",
        "Level_3": "Nephron_progenitor_cells"
    },
    "2": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    }
}
adata_1101_505396 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1101_505396, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(adata_1101_505396, color=["NPHS2", "DDN", "CUBN", "MAL", "CALB1"], cmap="viridis", use_raw=False)

# sc.pl.dotplot(adata_1101_505396, var_names=["CDX2", "FOXA1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1126_sciadv_abj5633_GSE165104_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
chosen_resolution = 0.5
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "3": {"Level_1": "Off_target_cells", "Level_2": "undifferentiated_pluripotent_cells", "Level_3": "undifferentiated_pluripotent_cells"},
    "9": {"Level_1": "Stromal_cells", "Level_2": "Stromal_progenitors", "Level_3": "Stromal_progenitors"},
    "5": {"Level_1": "Nephron", "Level_2": "Nephron_progenitor_cells", "Level_3": "Nephron_progenitor_cells"},
    "4": {"Level_1": "Nephron", "Level_2": "Nephron_progenitor_cells", "Level_3": "Nephron_progenitor_cells"},
    "6": {"Level_1": "Stromal_cells", "Level_2": "Stromal_progenitors", "Level_3": "Stromal_progenitors"},
    "12": {"Level_1": "Off_target_cells", "Level_2": "undifferentiated_pluripotent_cells", "Level_3": "undifferentiated_pluripotent_cells"},
    "7": {"Level_1": "Off_target_cells", "Level_2": "Off_target_cells", "Level_3": "Off_target_cells"}
}
adata_1126_sciadv_abj5633_GSE165104 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
sc.pl.violin(
    adata_1126_sciadv_abj5633_GSE165104,
    keys=["nCount_RNA", "nFeature_RNA"],
    groupby=dynamic_cluster_key
)
bad_clusters = ["0", "1"]
adata_1126_sciadv_abj5633_GSE165104 = adata_1126_sciadv_abj5633_GSE165104[
    ~adata_1126_sciadv_abj5633_GSE165104.obs[dynamic_cluster_key].isin(bad_clusters)
].copy()
for cluster_id, annotations in curated_overrides.items():
    mask = adata_1126_sciadv_abj5633_GSE165104.obs[dynamic_cluster_key].astype(str) == cluster_id
    for level, annotation in annotations.items():
        if level in adata_1126_sciadv_abj5633_GSE165104.obs.columns:
            adata_1126_sciadv_abj5633_GSE165104.obs.loc[mask, level] = annotation
if "Level_3_latest" not in adata_1126_sciadv_abj5633_GSE165104.obs.columns:
    adata_1126_sciadv_abj5633_GSE165104.obs["Level_3_latest"] = adata_1126_sciadv_abj5633_GSE165104.obs.get("Level_3", adata_1126_sciadv_abj5633_GSE165104.obs[dynamic_cluster_key])
for col in ["Level_1", "Level_2", "Level_3", "Level_3_latest"]:
    if col in adata_1126_sciadv_abj5633_GSE165104.obs.columns:
        adata_1126_sciadv_abj5633_GSE165104.obs[col] = adata_1126_sciadv_abj5633_GSE165104.obs[col].astype("category")
_plot_annotation_umaps_with_labels(
    adata_1126_sciadv_abj5633_GSE165104,
    ["Level_1", "Level_2", "Level_3_latest"],
    title_prefix=f"{dataset_name}_cleaned"
)
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / f"{Path(dataset_name).stem}_cleaned_annotated.h5ad"
adata_1126_sciadv_abj5633_GSE165104.write_h5ad(out_path)
if show_markers_tables:
    display_ranked_markers(adata_1126_sciadv_abj5633_GSE165104, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(
    adata_1126_sciadv_abj5633_GSE165104,
    color=["LYPD1", "NPHS2", "PODXL", "COL14A1", "PDGFRA", "FOXD1", "CUBN", "MAFB", "SIX2", "EYA1"],
    cmap="viridis",
    use_raw=False
)


In [ ]:
dataset_name = "d10_1172_jci_insight_122697_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1172_jci_insight_122697
chosen_resolution = 0.5
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "6": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    }
}
adata_1172_jci_insight_122697 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1172_jci_insight_122697, cluster_key=dynamic_cluster_key, n_genes=15)

# sc.pl.umap(adata_1172_jci_insight_122697, color=["CDX2", "FOXA1", "RET", "AQP2"], cmap="viridis", use_raw=False)
# sc.pl.dotplot(adata_1172_jci_insight_122697, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "d10_1242_dev_200198_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_1242_dev_200198
chosen_resolution = 0.4
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "8": {
        "Level_1": "Nephron",
        "Level_2": "Proximal_tubule",
        "Level_3": "Dev_PT"
    },
    "11": {
        "Level_1": "Stromal_cells",
        "Level_2": "Proliferative_stroma",
        "Level_3": "Proliferative_stroma"
    },
}
adata_1242_dev_200198 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_1242_dev_200198, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(adata_1242_dev_200198, color=["NPHS2", "CUBN", "LRP2", "LYPD1", "CALB1", "CLDN1"], cmap="viridis", use_raw=False)

# sc.pl.dotplot(adata_1242_dev_200198, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


In [ ]:
dataset_name = "dno_doi_kidney_organoid_GSE312980_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_kidney_organoid_GSE312980
chosen_resolution = 1.0
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = {
    "5": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "3": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "17": {
        "Level_1": "Ureteric_Epithelium",
        "Level_2": "Collecting_duct",
        "Level_3": "Collecting_duct"
    },
    "10": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "14": {
        "Level_1": "Stromal_cells",
        "Level_2": "Proliferative_stroma",
        "Level_3": "Proliferative_stroma"
    },
    "7": {
        "Level_1": "Off_target_cells",
        "Level_2": "Neural_progenitor_cells",
        "Level_3": "Neural_progenitor_cells"
    },
    "8": {
        "Level_1": "Stromal_cells",
        "Level_2": "Interstitial_Fibroblasts",
        "Level_3": "Interstitial_Fibroblasts"
    }
}
adata_kidney_organoid_GSE312980 = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_kidney_organoid_GSE312980, cluster_key=dynamic_cluster_key, n_genes=15)
adata_kidney_organoid_GSE312980.write_h5ad(out_path)
print(f"\nSaved final annotated object -> {out_path}")
sc.pl.umap(
    adata_kidney_organoid_GSE312980,
    color=["NPHS2", "CUBN", "LRP2", "CALB1", "NEFL"],
    cmap="viridis",
    use_raw=False
)

# sc.pl.dotplot(
#     adata_kidney_organoid_GSE312980,
#     var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"],
#     groupby=dynamic_cluster_key,
#     use_raw=False
# )


In [ ]:
dataset_name = "dno_doi_kidney_organoid_LMRH_harmonized_singlets_filtered.h5ad"
target_file = str(BASE_DIR / dataset_name)
#sym:adata_kidney_organoid_LMRH
chosen_resolution = 0.6
show_markers_tables = True
dynamic_cluster_key = f"leiden_res_{chosen_resolution}"
curated_overrides = None
{
    "11": {
        "Level_1": "Off_target_cells",
        "Level_2": "Endodermal_cells",
        "Level_3": "Endodermal_cells"
    }
}
adata_kidney_organoid_LMRH = process_and_annotate_dataset(
    file_path=target_file,
    marker_dict=MARKER_HIERARCHY,
    resolution=chosen_resolution,
    cluster_key=dynamic_cluster_key,
    manual_annotations=curated_overrides
)
if show_markers_tables:
    display_ranked_markers(adata_kidney_organoid_LMRH, cluster_key=dynamic_cluster_key, n_genes=15)
sc.pl.umap(adata_kidney_organoid_LMRH, color=["CDX2", "SI", "CDH17", "AGR2", "APOA1", "TTR", "FABP1", "GSTA1", "SHH", "GATA4"], cmap="viridis", use_raw=False)

# sc.pl.dotplot(adata_kidney_organoid_LMRH, var_names=["CDX2", "FOXA1", "KRT19", "CDH1", "RET", "AQP2"], groupby=dynamic_cluster_key, use_raw=False)


## Cross-dataset summary

Figures and CSV tables from all `*_annotated.h5ad` files in the output folder.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")
SUMMARY_OUT.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR = OUTPUT_DIR
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": FIGURE_DPI,
    "savefig.dpi": FIGURE_DPI,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
})
LEVEL1_PALETTE = {
    "Nephron":              "#2166AC",
    "Stromal_cells":        "#D6604D",
    "Endothelial_cells":    "#4DAC26",
    "Ureteric_Epithelium":  "#8E44AD",
    "Off_target_cells":     "#B0B0B0",
}
LEVEL2_PALETTE = {
    "Nephron_progenitor_cells":  "#084594",
    "Podocytes":                 "#2171B5",
    "Proximal_tubule":           "#4292C6",
    "Proliferative_tubule":      "#6BAED6",
    "Distal_tubule":             "#9ECAE1",
    "Loop_of_Henle":             "#C6DBEF",
    "Stromal_progenitors":       "#CB181D",
    "Proliferative_stroma":      "#EF3B2C",
    "Interstitial_Fibroblasts":  "#FB6A4A",
    "Mesangial_cells":           "#FC9272",
    "Early_endothelium":         "#238B45",
    "Glomerular_endothelium":    "#41AB5D",
    "Lymphatic_endothelium":     "#74C476",
    "Collecting_duct":           "#6A51A3",
    "Ureteric_tip":              "#9E9AC8",
    "Neural_progenitor_cells":   "#969696",
    "Glial_cells":               "#BDBDBD",
    "Muscle_progenitor_cells":   "#D9D9D9",
    "Endodermal_cells":          "#737373",
    "Immune_cells":              "#525252",
}
print("-" * 70)
print("LOADING ANNOTATED DATASETS")
print("-" * 70)
h5ad_files = sorted(ANNOTATED_DIR.glob("*_annotated.h5ad"))
if not h5ad_files:
    raise FileNotFoundError(
        f"No *_annotated.h5ad files found in:\n{ANNOTATED_DIR}\n"
        "Please verify the OUTPUT_DIR path above."
    )
records_l1 = []   # one row per cell
records_l2 = []
dataset_meta = []
for fpath in h5ad_files:
    label = fpath.stem.replace("_annotated", "")
    adata = sc.read_h5ad(fpath)
    n_cells = adata.n_obs
    print(f"+ {label:<44} {n_cells:>7,} cells")

    # Resolve annotation columns (handle capitalisation variants)
    obs_cols = {c.lower(): c for c in adata.obs.columns}
    col_l1 = obs_cols.get("level_1") or obs_cols.get("level1")
    col_l2 = obs_cols.get("level_2") or obs_cols.get("level2")
    if col_l1 is None:
        print(f"[!] Level_1 column missing - skipping {label}")
        continue
    l1_series = adata.obs[col_l1].astype(str)
    l2_series = adata.obs[col_l2].astype(str) if col_l2 else pd.Series(
        ["Unknown"] * n_cells, index=adata.obs_names
    )
    for ct, grp in l1_series.groupby(l1_series):
        records_l1.append({"dataset": label, "Level_1": ct, "n_cells": len(grp)})
    for ct, grp in l2_series.groupby(l2_series):
        l1_val = l1_series[grp.index].mode()[0] if len(grp) > 0 else "Unknown"
        records_l2.append({
            "dataset": label, "Level_2": ct,
            "Level_1": l1_val, "n_cells": len(grp)
        })
    dataset_meta.append({
        "dataset": label,
        "filename": fpath.name,
        "n_cells": n_cells,
        "n_clusters": l1_series.nunique(),
    })
df_l1 = pd.DataFrame(records_l1)
df_l2 = pd.DataFrame(records_l2)
df_meta = pd.DataFrame(dataset_meta).set_index("dataset")
df_l1_pct = (
    df_l1.pivot_table(index="dataset", columns="Level_1",
                      values="n_cells", aggfunc="sum", fill_value=0)
)
df_l1_pct = df_l1_pct.div(df_l1_pct.sum(axis=1), axis=0) * 100
df_l1_pct = df_l1_pct.reindex(columns=list(LEVEL1_PALETTE.keys()), fill_value=0)
df_l2_pct = (
    df_l2.pivot_table(index="dataset", columns="Level_2",
                      values="n_cells", aggfunc="sum", fill_value=0)
)
df_l2_pct = df_l2_pct.div(df_l2_pct.sum(axis=1), axis=0) * 100
sort_order = df_l1_pct.get("Nephron", pd.Series(0, index=df_l1_pct.index)) \
                       .sort_values(ascending=False).index.tolist()
df_l1_pct = df_l1_pct.loc[sort_order]
df_l2_pct = df_l2_pct.reindex(sort_order)
print(f"\nDatasets loaded: {len(df_meta)}")
print(f"Total cells: {df_meta['n_cells'].sum():,}")
print("\nRendering Figure 1 - Level-1 Composition ...")
n_ds = len(df_l1_pct)
fig1, ax1 = plt.subplots(figsize=(max(12, n_ds * 0.45), 8))
bar_colors = [LEVEL1_PALETTE.get(c, "#AAAAAA") for c in df_l1_pct.columns]
bottom = np.zeros(n_ds)
x_pos = np.arange(n_ds)
for col, color in zip(df_l1_pct.columns, bar_colors):
    vals = df_l1_pct[col].values
    bars = ax1.bar(x_pos, vals, bottom=bottom, color=color,
                  width=1.0, edgecolor="white", linewidth=0.4)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 8:
            ax1.text(i, b + v / 2, f"{v:.0f}%",
                    ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold")
    bottom += vals
ax1.set_xticks(x_pos)
ax1.set_xticklabels(df_l1_pct.index, fontsize=9, rotation=45, ha="right")
ax1.set_ylabel("Percentage of Cells (%)", fontsize=10)
ax1.set_ylim(0, 100)
ax1.set_title(
    "Cell-Type Composition Across Kidney Organoid Datasets\n"
    "Snapseed Hierarchical Annotation - Level 1",
    fontsize=13, fontweight="bold", pad=12
)
ax1.yaxis.set_major_locator(MaxNLocator(10))
ax1.tick_params(axis="y", labelsize=9)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.tick_params(axis="x", length=0)
legend_patches = [
    mpatches.Patch(facecolor=LEVEL1_PALETTE[ct], label=ct.replace("_", ""),
                   edgecolor="#555555", linewidth=0.5)
    for ct in df_l1_pct.columns if ct in LEVEL1_PALETTE
]
ax1.legend(handles=legend_patches, loc="upper right", bbox_to_anchor=(1.25, 1),
          frameon=True, framealpha=0.92, edgecolor="#CCCCCC",
          fontsize=9, title="Cell Lineage", title_fontsize=9)
ax_twin = ax1.twiny()
ax_twin.set_xlim(ax1.get_xlim())
ax_twin.set_xticks(x_pos)
cell_counts = [df_meta.loc[ds, "n_cells"] if ds in df_meta.index else 0
               for ds in df_l1_pct.index]
ax_twin.set_xticklabels(
    [f"n = {c:,}" for c in cell_counts],
    fontsize=8, color="#555555", rotation=45, ha="left"
)
ax_twin.tick_params(axis="x", length=0)
ax_twin.spines["top"].set_visible(False)
ax_twin.spines["bottom"].set_visible(False)
ax_twin.spines["left"].set_visible(False)
ax_twin.spines["right"].set_visible(False)
plt.tight_layout()
fig1.savefig(SUMMARY_OUT / "Fig1_Level1_Composition_BarChart.pdf")
fig1.savefig(SUMMARY_OUT / "Fig1_Level1_Composition_BarChart.png")
plt.show()
print("Figure 1 saved")
print("Rendering Figure 3 - Consistency Matrix ...")
keep_l2 = [c for c in df_l2_pct.columns
           if (df_l2_pct[c] > 0).sum() >= 3
           and c not in ("nan", "None", "Unknown", "Unassigned")]
detect_mat = (df_l2_pct[keep_l2] >= 1.0).astype(int)
n_detected = detect_mat.sum(axis=0).sort_values(ascending=False)
detect_sorted = detect_mat[n_detected.index]
cmap_bin = LinearSegmentedColormap.from_list(
    "presence", ["#F0F0F0", "#2166AC"], N=2
)
fig3, ax3 = plt.subplots(figsize=(max(14, len(keep_l2) * 0.82), max(6, n_ds * 0.42)))
ax3.imshow(detect_sorted.values, cmap=cmap_bin, aspect="auto",
           vmin=0, vmax=1, interpolation="nearest")
for x in range(detect_sorted.shape[1] + 1):
    ax3.axvline(x - 0.5, color="white", linewidth=0.5)
for y in range(detect_sorted.shape[0] + 1):
    ax3.axhline(y - 0.5, color="white", linewidth=0.5)
for i in range(detect_sorted.shape[0]):
    for j in range(detect_sorted.shape[1]):
        if detect_sorted.values[i, j] == 1:
            ax3.text(j, i, "+", ha="center", va="center",
                     fontsize=9, color="white", fontweight="bold")
ax3.set_xticks(range(len(detect_sorted.columns)))
ax3.set_xticklabels([c.replace("_", "\n") for c in detect_sorted.columns],
                    fontsize=8, rotation=90, ha="center")
ax3.set_yticks(range(n_ds))
ax3.set_yticklabels(detect_sorted.index, fontsize=9)
ax3.set_title(
    "Cell-Type Detection Consistency Across Datasets\n"
    "+ = detected in ≥ 1% of cells - Sorted by prevalence",
    fontsize=13, fontweight="bold", pad=15
)
plt.tight_layout()
fig3.savefig(SUMMARY_OUT / "Fig3_CellType_Consistency_Matrix.pdf")
fig3.savefig(SUMMARY_OUT / "Fig3_CellType_Consistency_Matrix.png")
plt.show()
print("Figure 3 saved")
print("Rendering Figure 4 Panels - Summary Statistics ...")
fig4a, ax4a = plt.subplots(figsize=(10, 6))
ds_counts = df_meta["n_cells"].reindex(sort_order).dropna()
colors_4a = plt.cm.Blues(np.linspace(0.35, 0.85, len(ds_counts)))[::-1]
ax4a.bar(
    range(len(ds_counts)),
    ds_counts.values / 1000,
    color=colors_4a, edgecolor="white", linewidth=0.4, width=0.72
)
ax4a.set_xticks(range(len(ds_counts)))
ax4a.set_xticklabels(ds_counts.index, fontsize=8, rotation=45, ha="right")
ax4a.set_ylabel("Cells (×10³)", fontsize=9)
ax4a.set_title("Dataset Size", fontsize=12, fontweight="bold")
ax4a.tick_params(axis="x", length=0)
ax4a.spines["top"].set_visible(False)
ax4a.spines["right"].set_visible(False)
for i, v in enumerate(ds_counts.values):
    ax4a.text(i, v / 1000 + 0.5, f"{v/1000:.1f}k",
              ha="center", fontsize=7.5, color="#333333")
fig4a.savefig(SUMMARY_OUT / "Fig4A_Dataset_Size.pdf", bbox_inches="tight")
fig4a.savefig(SUMMARY_OUT / "Fig4A_Dataset_Size.png", bbox_inches="tight")
plt.show()
print("Figure 4A saved")
fig4b, ax4b = plt.subplots(figsize=(7, 7))
mean_l1 = df_l1_pct.mean()
mean_l1 = mean_l1[mean_l1 > 0.1]
pie_colors = [LEVEL1_PALETTE.get(c, "#AAAAAA") for c in mean_l1.index]
wedges, texts, autotexts = ax4b.pie(
    mean_l1.values,
    labels=[c.replace("_", "") for c in mean_l1.index],
    colors=pie_colors,
    autopct=lambda p: f"{p:.1f}%" if p > 4 else "",
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 10}
)
for at in autotexts:
    at.set_fontsize(9.5)
    at.set_fontweight("bold")
    at.set_color("white")
ax4b.set_title("Mean Composition\n(All Datasets)", fontsize=12, fontweight="bold")
fig4b.savefig(SUMMARY_OUT / "Fig4B_Mean_Composition.pdf", bbox_inches="tight")
fig4b.savefig(SUMMARY_OUT / "Fig4B_Mean_Composition.png", bbox_inches="tight")
plt.show()
print("Figure 4B saved")
fig4c, ax4c = plt.subplots(figsize=(10, 6))
detect_freq = detect_mat.sum(axis=0).sort_values(ascending=True)
bar_colors_4c = [LEVEL2_PALETTE.get(c, "#AAAAAA") for c in detect_freq.index]
ax4c.bar(range(len(detect_freq)), detect_freq.values,
         color=bar_colors_4c, edgecolor="white", linewidth=0.4, width=0.72)
ax4c.axhline(len(df_meta) * 0.5, color="#CC0000", linestyle="--",
             linewidth=1.2, alpha=0.8, label="50% threshold")
ax4c.set_xticks(range(len(detect_freq)))
ax4c.set_xticklabels([c.replace("_", "") for c in detect_freq.index], fontsize=8, rotation=90)
ax4c.set_ylabel("Number of Datasets", fontsize=9)
ax4c.set_title("Cell-Type Detection Frequency", fontsize=12, fontweight="bold")
ax4c.legend(fontsize=9, framealpha=0.85)
ax4c.tick_params(axis="x", length=0)
ax4c.spines["top"].set_visible(False)
ax4c.spines["right"].set_visible(False)
for i, v in enumerate(detect_freq.values):
    ax4c.text(i, v + 0.3, str(v), ha="center", fontsize=8)
fig4c.savefig(SUMMARY_OUT / "Fig4C_Detection_Frequency.pdf", bbox_inches="tight")
fig4c.savefig(SUMMARY_OUT / "Fig4C_Detection_Frequency.png", bbox_inches="tight")
plt.show()
print("Figure 4C saved")
fig4d, ax4d = plt.subplots(figsize=(8, 7))
x_neph = df_l1_pct.get("Nephron", pd.Series(0, index=df_l1_pct.index))
y_stro = df_l1_pct.get("Stromal_cells", pd.Series(0, index=df_l1_pct.index))
s_cells = df_meta["n_cells"].reindex(x_neph.index).fillna(1000)
sc4d = ax4d.scatter(
    x_neph, y_stro,
    s=s_cells / 800,
    c=s_cells,
    cmap="YlOrRd",
    alpha=0.8, edgecolors="#555555", linewidths=0.5, zorder=3
)
for ds in x_neph.index:
    short = ds[:18]
    ax4d.annotate(short, (x_neph[ds], y_stro[ds]),
                  fontsize=7, alpha=0.8,
                  xytext=(4, 4), textcoords="offset points")
ax4d.set_xlabel("Nephron (%)", fontsize=10)
ax4d.set_ylabel("Stromal Cells (%)", fontsize=10)
ax4d.set_title("Nephron vs. Stromal Balance\n(bubble size = dataset size)",
               fontsize=12, fontweight="bold")
ax4d.grid(True, linestyle="--", alpha=0.3)
cb4d = fig4d.colorbar(sc4d, ax=ax4d, fraction=0.04, pad=0.02)
cb4d.set_label("Total cells", fontsize=9)
cb4d.ax.tick_params(labelsize=8)
fig4d.savefig(SUMMARY_OUT / "Fig4D_Nephron_Stromal_Scatter.pdf", bbox_inches="tight")
fig4d.savefig(SUMMARY_OUT / "Fig4D_Nephron_Stromal_Scatter.png", bbox_inches="tight")
plt.show()
print("Figure 4D saved")
print("\nExporting CSV tables ...")
l1_types = list(LEVEL1_PALETTE.keys())
l2_to_l1 = {}
for _, row in df_l2.drop_duplicates("Level_2").iterrows():
    l2_to_l1[row["Level_2"]] = row["Level_1"]
tbl1 = df_l1_pct.round(2).copy()
tbl1.insert(0, "total_cells", df_meta["n_cells"].reindex(tbl1.index))
tbl1.index.name = "dataset"
tbl1.to_csv(SUMMARY_OUT / "Table1_Level1_Composition_Percent.csv")
print("Table1_Level1_Composition_Percent.csv")
tbl2 = df_l2_pct.round(2).copy()
tbl2.insert(0, "total_cells", df_meta["n_cells"].reindex(tbl2.index))
tbl2.index.name = "dataset"
tbl2.to_csv(SUMMARY_OUT / "Table2_Level2_Composition_Percent.csv")
print("Table2_Level2_Composition_Percent.csv")
raw_l1 = df_l1.pivot_table(index="dataset", columns="Level_1",
                            values="n_cells", aggfunc="sum", fill_value=0)
raw_l1.insert(0, "total_cells", raw_l1.sum(axis=1))
raw_l1.index.name = "dataset"
raw_l1.to_csv(SUMMARY_OUT / "Table3_Level1_CellCounts_Raw.csv")
print("Table3_Level1_CellCounts_Raw.csv")
detect_mat.index.name = "dataset"
detect_mat.to_csv(SUMMARY_OUT / "Table4_Level2_Consistency_Binary.csv")
print("Table4_Level2_Consistency_Binary.csv")
summary_rows = []
for ct in keep_l2:
    l1_parent = l2_to_l1.get(ct, "Unknown")
    pct_vals = df_l2_pct[ct].values
    n_present = int((pct_vals >= 1).sum())
    summary_rows.append({
        "Level_2_CellType":      ct,
        "Parent_Lineage":        l1_parent,
        "Datasets_Detected":     n_present,
        "Detection_Rate_%":      round(n_present / len(df_meta) * 100, 1),
        "Mean_Fraction_%":       round(float(pct_vals[pct_vals >= 1].mean()), 2)
                                 if n_present > 0 else 0,
        "Median_Fraction_%":     round(float(np.median(pct_vals[pct_vals >= 1])), 2)
                                 if n_present > 0 else 0,
        "Max_Fraction_%":        round(float(pct_vals.max()), 2),
        "Std_Fraction_%":        round(float(pct_vals[pct_vals >= 1].std()), 2)
                                 if n_present > 1 else 0,
    })
tbl5 = pd.DataFrame(summary_rows).sort_values(
    ["Parent_Lineage", "Datasets_Detected"], ascending=[True, False]
)
tbl5.to_csv(SUMMARY_OUT / "Table5_GlobalCellType_Summary.csv", index=False)
print("Table5_GlobalCellType_Summary.csv")
df_meta.reset_index().to_csv(SUMMARY_OUT / "Table6_Dataset_Metadata.csv", index=False)
print("Table6_Dataset_Metadata.csv")
print()
print("-" * 70)
print("ANNOTATION SUMMARY REPORT")
print("-" * 70)
print(f"Datasets annotated : {len(df_meta)}")
print(f"Total cells processed : {df_meta['n_cells'].sum():,}")
print(f"Level-1 lineages : {len(l1_types)}")
print(f"Level-2 cell types : {len(keep_l2)}")
print()
print("--- Level-1 lineage distribution (mean across datasets) ---")
for ct in l1_types:
    if ct in df_l1_pct.columns:
        m = df_l1_pct[ct].mean()
        bar = "#" * int(m / 2)
        print(f"{ct:<28} {m:5.1f}% {bar}")
print()
print("--- Universally detected cell types (≥ 80% datasets) ---─")
universal = tbl5[tbl5["Detection_Rate_%"] >= 80][
    ["Level_2_CellType", "Parent_Lineage", "Detection_Rate_%", "Mean_Fraction_%"]
]
display(universal.reset_index(drop=True))
print()
print(f"Figures saved to : {SUMMARY_OUT}")
print("-" * 70)


## Protocol composition

Nephron vs stromal balance by differentiation protocol (`diff_protocol` in obs metadata).


In [ ]:
SUMMARY_OUT.mkdir(parents=True, exist_ok=True)
INPUT_DIR = OUTPUT_DIR

warnings.filterwarnings("ignore", category=UserWarning, module="anndata")
SUMMARY_OUT.mkdir(parents=True, exist_ok=True)
h5ad_files = sorted(INPUT_DIR.glob("*.h5ad"))
if not h5ad_files:
    raise FileNotFoundError(f"No .h5ad files found in: {INPUT_DIR}")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": FIGURE_DPI
})

def extract_protocol_from_metadata(obs_df):
    if "diff_protocol" not in obs_df.columns:
        raise KeyError(
            f"No diff_protocol metadata column found in obs. Available columns: {list(obs_df.columns)}"
        )
    protocol_col = "diff_protocol"
    protocol_series = obs_df[protocol_col].dropna().astype(str).str.strip()
    protocol = protocol_series.mode().iat[0] if not protocol_series.empty else "Unknown"
    return protocol, protocol_col

def get_protocol_color(protocol, other_idx):
    """Assigns strict, distinct colors based on protocol families."""
    p = str(protocol).lower().strip()
    if p == "morizane":
        return "#08519C"
    if p == "morizane_modified":
        return "#6BAED6"
    if p == "takasato":
        return "#E31A1C"
    if p == "takasato_modified":
        return "#FB6A4A"
    if "hybrid_takasato_morizane" in p:
        return "#7A0177"
    if "hybrid_takasato_uchimura" in p:
        return "#CE1256"
    other_palette = [
        "#33A02C",
        "#FF7F00",
        "#B15928",
        "#17BECF",
        "#B2DF8A",
        "#999999",
    ]
    return other_palette[other_idx % len(other_palette)]
print(f"Extracting profiles from {len(h5ad_files)} annotated datasets...")
records = []
override_targets = {
    "d10_1016_j_stem_2018_10_010": "Hybrid_Takasato_Morizane",
}
for fpath in h5ad_files:
    dataset_label = (
        fpath.stem.replace("_annotated", "")
        .replace("_harmonized_singlets_filtered", "")
        .replace("_cellbender_feature_bc_matrix_singlets_filtered", "")
    )
    try:
        adata = sc.read_h5ad(fpath, backed="r")
        obs_df = adata.obs
        obs_cols = {c.lower(): c for c in obs_df.columns}
        l1_key = obs_cols.get("level_1") or obs_cols.get("level1")
        if not l1_key:
            print(f"[!] Missing Level_1 annotations in {fpath.name}")
            continue
        protocol, protocol_col = extract_protocol_from_metadata(obs_df)
        for target_key, new_protocol in override_targets.items():
            if target_key in fpath.name:
                protocol = new_protocol
                print(f"[override] matched '{target_key}' in {fpath.name} -> {protocol}")
                break # Stop searching once a match is found
        total_cells = len(obs_df)
        counts = obs_df[l1_key].value_counts()
        nephron_count = counts.get("Nephron", 0)
        stromal_count = counts.get("Stromal_cells", counts.get("Stromal", 0))
        records.append({
            "Dataset": dataset_label,
            "Protocol": protocol,
            "Nephron_pct": (nephron_count / total_cells) * 100,
            "Stromal_pct": (stromal_count / total_cells) * 100,
            "Total_Cells": total_cells
        })
    except Exception as e:
        print(f"[!] Error reading {fpath.name}: {e}")
df_plot = pd.DataFrame(records)
if not df_plot.empty:
    display(df_plot[df_plot["Dataset"].str.contains("d10_1016_j_stem_2018_10_010", na=False)])
unique_protocols = sorted(df_plot["Protocol"].unique())
protocol_cmap = {}
other_counter = 0
for proto in unique_protocols:
    if any(fam in proto.lower() for fam in ["morizane", "takasato"]):
        protocol_cmap[proto] = get_protocol_color(proto, 0)
    else:
        protocol_cmap[proto] = get_protocol_color(proto, other_counter)
        other_counter += 1
print("\nRendering Protocol Composition Scatter Plot...")
fig, ax = plt.subplots(figsize=(10, 8))
for proto, group in df_plot.groupby("Protocol"):
    ax.scatter(
        group["Nephron_pct"],
        group["Stromal_pct"],
        color=protocol_cmap.get(proto, "#B0B0B0"),
        label=proto,
        s=120,
        alpha=0.9,
        edgecolors="white",
        linewidths=0.8,
        zorder=3
    )
lmrh_row = df_plot[df_plot["Dataset"].str.contains("LMRH", case=False, na=False)].iloc[0]
dx = -2
dy = 10
ax.annotate(
    "LMRH",
    xy=(lmrh_row["Nephron_pct"], lmrh_row["Stromal_pct"]),
    xytext=(lmrh_row["Nephron_pct"] + dx, lmrh_row["Stromal_pct"] + dy),
    fontsize=9,
    ha='left' if dx > 0 else 'right',
    va='bottom' if dy > 0 else 'top',
    arrowprops=dict(
        arrowstyle='->',
        color='gray',
        lw=0.8,
        shrinkA=8,
        shrinkB=4
    )
)
ax.set_xlabel("Nephron Lineage Proportion (%)", fontweight="bold", fontsize=11)
ax.set_ylabel("Stromal Lineage Proportion (%)", fontweight="bold", fontsize=11)
ax.set_title(
    "Nephron vs. Stromal Lineage Balance Across Datasets\n"
    "Colored by Differentiation Protocol",
    fontsize=13,
    fontweight="bold",
    pad=15
)
ax.set_xlim(-2, max(df_plot["Nephron_pct"].max() + 5, 100))
ax.set_ylim(-2, max(df_plot["Stromal_pct"].max() + 5, 100))
ax.grid(True, linestyle="--", alpha=0.4, zorder=1)
ax.legend(
    title="Diff Protocol",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    frameon=True,
    edgecolor="#CCCCCC",
    fontsize=9
)
plt.tight_layout()
fig.savefig(SUMMARY_OUT / "Protocol_Balance_Scatter_MetadataApplied.pdf", bbox_inches="tight")
fig.savefig(SUMMARY_OUT / "Protocol_Balance_Scatter_MetadataApplied.png", bbox_inches="tight", dpi=FIGURE_DPI)
plt.show()
print(f"Plot saved to: {SUMMARY_OUT.name}/Protocol_Balance_Scatter_MetadataApplied.png")


In [ ]:
print("\nRendering Protocol Composition Scatter Plot with Custom Biological Clouds...")
fig, ax = plt.subplots(figsize=(11, 8.5))

import matplotlib.patches as mpatches
import matplotlib.path as mpath
from scipy.interpolate import splprep, splev
takasato_core = df_plot[
    (df_plot['Protocol'].str.lower() == 'takasato') |
    (df_plot['Protocol'].str.lower().str.contains('hybrid_takasato_uchimura'))
]
morizane_core_filtered = df_plot[
    (
        df_plot['Protocol'].str.lower().str.contains('morizane') |
        df_plot['Protocol'].str.lower().str.contains('hybrid_takasato_morizane') |
        (df_plot['Protocol'].str.lower() == 'vanslambrouck_2022')
    ) &
    (~df_plot['Dataset'].str.contains('2022_04_009')) & (~df_plot['Dataset'].str.contains('2021_12_010')) # <-- Explicit exclusion
]

def draw_smooth_organic_cloud(df, color, label_text, pad=5):
    if len(df) >= 3:
        points = df[['Nephron_pct', 'Stromal_pct']].values
        from scipy.spatial import ConvexHull
        hull = ConvexHull(points)
        hull_points = points[hull.vertices]
        hull_points = np.vstack([hull_points, hull_points[0]])
        cx, cy = np.mean(points[:, 0]), np.mean(points[:, 1])
        padded_points = []
        for p in hull_points:
            vec = p - [cx, cy]
            norm = np.linalg.norm(vec)
            padded_p = p + (vec / norm) * pad if norm > 0 else p
            padded_points.append(padded_p)
        padded_points = np.array(padded_points)
        x, y = padded_points[:, 0], padded_points[:, 1]
        tck, u = splprep([x, y], s=0, per=True)
        u_new = np.linspace(0, 1, 200)
        smooth_x, smooth_y = splev(u_new, tck)
        smooth_vertices = np.column_stack([smooth_x, smooth_y])
        poly = mpatches.Polygon(smooth_vertices, closed=True, facecolor=color, edgecolor=color,
                                lw=1.5, ls='-', alpha=0.12, zorder=1, label=f'{label_text} Domain')
        ax.add_patch(poly)
        poly_outline = mpatches.Polygon(smooth_vertices, closed=True, facecolor='none', edgecolor=color,
                                        lw=8, alpha=0.35, zorder=2)
        ax.add_patch(poly_outline)
draw_smooth_organic_cloud(takasato_core, '#E31A1C', 'Classical Takasato')
draw_smooth_organic_cloud(morizane_core_filtered, '#08519C', 'Morizane Axis Core')
for proto, group in df_plot.groupby("Protocol"):
    ax.scatter(
        group["Nephron_pct"],
        group["Stromal_pct"],
        color=protocol_cmap.get(proto, "#B0B0B0"),
        label=proto,
        s=140,
        alpha=0.95,
        edgecolors="white",
        linewidths=0.9,
        zorder=4
    )
lmrh_row = df_plot[df_plot["Dataset"].str.contains("LMRH", case=False, na=False)].iloc[0]
dx = -2
dy = 10
ax.annotate(
    "LMRH",
    xy=(lmrh_row["Nephron_pct"], lmrh_row["Stromal_pct"]),
    xytext=(lmrh_row["Nephron_pct"] + dx, lmrh_row["Stromal_pct"] + dy),
    fontsize=9,
    ha='left' if dx > 0 else 'right',
    va='bottom' if dy > 0 else 'top',
    arrowprops=dict(
        arrowstyle='->',
        color='gray',
        lw=0.8,
        shrinkA=8,
        shrinkB=4
    )
)
ax.set_xlabel("Nephron Lineage Proportion (%)", fontweight="bold", fontsize=11, labelpad=10)
ax.set_ylabel("Stromal Lineage Proportion (%)", fontweight="bold", fontsize=11, labelpad=10)
ax.set_title(
    "Lineage Segregation: Takasato vs. Morizane Protocol Architectures\n"
    "Delineated Domains Isolate Protocol Modifications from Family Archetypes",
    fontsize=13, fontweight="bold", pad=20
)
ax.set_xlim(-3, 103)
ax.set_ylim(-3, 103)
ax.grid(True, linestyle=":", alpha=0.5, zorder=0)
ax.legend(
    title="Diff Protocol & Domains",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    frameon=True,
    edgecolor="#CCCCCC",
    fontsize=9
)
plt.tight_layout()
fig.savefig(SUMMARY_OUT / "Protocol_Balance_Scatter_CustomClouds.png", bbox_inches="tight", dpi=FIGURE_DPI)
plt.show()


## Adult/fetal pseudobulk correlation

Spearman correlation of organoid Level 3 profiles against adult and fetal reference atlases.


In [ ]:
warnings.filterwarnings("ignore")
HARMONIZATION_MAP = {
    "podocytes": "podocyte", "mature_podocytes": "podocyte", "precursor_podocytes": "podocyte", "podocyte": "podocyte",
    "proximal_tubule": "proximal_tubule", "pt": "proximal_tubule", "dev_pt": "proximal_tubule", "epithelial_cell_of_proximal_tubule": "proximal_tubule",
    "distal_tubule": "distal_tubule", "dt": "distal_tubule", "kidney_distal_convoluted_tubule_epithelial_cell": "distal_tubule",
    "loh": "loop_of_henle", "kidney_loop_of_henle": "loop_of_henle", "adult_kidney loop of henle thick ascending limb epithelial cell": "loop_of_henle",
    "adult_kidney loop of henle thin ascending limb epithelial cell": "loop_of_henle", "adult_kidney loop of henle thin descending limb epithelial cell": "loop_of_henle",
    "collecting_duct": "collecting_duct", "kidney_inner_medulla_collecting_duct_epithelial_cell": "collecting_duct", "kidney_connecting_tubule_epithelial_cell": "collecting_duct",
    "collecting_duct_intercalated_cells": "cd_intercalated", "kidney_collecting_duct_intercalated_cell": "cd_intercalated",
    "collecting_duct_principal_cells": "cd_principal", "kidney_collecting_duct_principal_cell": "cd_principal",
    "endothelial_cells": "endothelial", "early_endothelium": "endothelial", "endothelial_cell": "endothelial", "adult_endothelial_cell": "endothelial",
    "adult_endothelial cell of lymphatic vessel": "endothelial", "adult_glomerular capillary endothelial cell": "endothelial",
    "adult_peritubular capillary endothelial cell": "endothelial", "adult_kidney arterial blood vessel cell": "endothelial", "vasa_recta_cell": "endothelial",
    "interstitial_fibroblasts": "fibroblast", "activated fibroblasts": "fibroblast", "stromal_cells": "fibroblast", "proliferative_stroma": "fibroblast",
    "stromal_progenitors": "fibroblast", "kidney_interstitial_fibroblast": "fibroblast", "renal_medullary_fibroblast": "fibroblast", "fetal_kidney interstitial cell": "fibroblast",
    "mesangial_cells": "mesangial", "fetal_glomerular mesangial cell": "mesangial",
    "pericytes": "mural_cell", "mural_cell": "mural_cell", "vascular_associated_smooth_muscle_cell": "mural_cell",
    "parietal_epithelial_cell": "parietal_epithelial", "mesonephric_nephron_tubule_epithelial_cell": "mesonephric_epithelial"
}

def to_canonical(raw_string):
    clean_str = str(raw_string).lower().strip()
    return HARMONIZATION_MAP.get(clean_str, clean_str)

def generate_pseudobulk(adata, celltype_col):
    """Collapses single-cell adata into a pseudobulk dataframe (mean expression)."""
    adata.obs['canonical_ct'] = adata.obs[celltype_col].apply(to_canonical)
    X = adata.X.toarray() if issparse(adata.X) else adata.X
    df = pd.DataFrame(X, index=adata.obs.index, columns=adata.var_names)
    df['canonical_ct'] = adata.obs['canonical_ct'].values
    return df.groupby('canonical_ct').mean()
print("Loading Reference Datasets...")
adata_adult = sc.read_h5ad(ADULT_REF_PATH)
adata_adult.var_names = adata_adult.var['feature_name'].astype(str)
adata_adult.var_names_make_unique() # Crucial: prevents duplicate symbol errors
pb_adult = generate_pseudobulk(adata_adult, celltype_col="cell_type")
adata_fetal = sc.read_h5ad(FETAL_REF_PATH)
adata_fetal.var_names = adata_fetal.var['feature_name'].astype(str)
adata_fetal.var_names_make_unique()
pb_fetal = generate_pseudobulk(adata_fetal, celltype_col="cell_type")
print("Loading Organoid Datasets...")
pb_organoids = {}
for fpath in sorted(ANNOTATED_DIR.glob("*_annotated.h5ad")):
    raw_name = fpath.stem
    terms_to_remove = ["_annotated", "harmonized", "filtered", "d10_1016"]
    for term in terms_to_remove:
        raw_name = raw_name.replace(term, "")
    dataset_name = re.sub(r'_+', '_', raw_name).strip('_')
    adata_org = sc.read_h5ad(fpath)
    obs_cols = {c.lower(): c for c in adata_org.obs.columns}
    c3 = obs_cols.get("level_3") or obs_cols.get("level3")
    if c3:
        pb_organoids[dataset_name] = generate_pseudobulk(adata_org, celltype_col=c3)
common_genes = set(pb_adult.columns).intersection(pb_fetal.columns)
for pb in pb_organoids.values():
    common_genes = common_genes.intersection(pb.columns)
common_genes = sorted(list(common_genes))
print(f"+ Setup Complete. Found {len(common_genes)} shared genes across all datasets.")


In [ ]:
var_fetal = pb_fetal[common_genes].var()
var_adult = pb_adult[common_genes].var()
combined_var = (var_fetal + var_adult) / 2
top_hvgs = combined_var.nlargest(2000).index.tolist()
results_hvg = []
for ds_name, pb_org in pb_organoids.items():
    shared_cts = set(pb_org.index).intersection(pb_adult.index).intersection(pb_fetal.index)
    ds_fetal_corrs, ds_adult_corrs = [], []
    for ct in shared_cts:
        vec_org = pb_org.loc[ct, top_hvgs].values
        vec_fetal = pb_fetal.loc[ct, top_hvgs].values
        vec_adult = pb_adult.loc[ct, top_hvgs].values
        corr_fetal, _ = spearmanr(vec_org, vec_fetal)
        corr_adult, _ = spearmanr(vec_org, vec_adult)
        ds_fetal_corrs.append(corr_fetal)
        ds_adult_corrs.append(corr_adult)
    if ds_fetal_corrs:
        results_hvg.append({
            "Dataset": ds_name,
            "Fetal_Spearman": np.mean(ds_fetal_corrs),
            "Adult_Spearman": np.mean(ds_adult_corrs)
        })
df_hvg = pd.DataFrame(results_hvg)
ref_corrs = []
for ct in set(pb_adult.index).intersection(pb_fetal.index):
    corr, _ = spearmanr(pb_adult.loc[ct, top_hvgs].values, pb_fetal.loc[ct, top_hvgs].values)
    ref_corrs.append(corr)
baseline_ref_corr = np.mean(ref_corrs)
DATASET_LABELS = {
    "LMRH": "LMRH"
}
plt.clf()
fig, ax = plt.subplots(figsize=(11, 8), dpi=FIGURE_DPI)
all_values = pd.concat([df_hvg['Fetal_Spearman'], df_hvg['Adult_Spearman']])
lims = [max(min(all_values.min(), baseline_ref_corr) - 0.1, 0), 1.05]
ax.plot(lims, lims, 'k--', alpha=0.3, zorder=1, label="Equal Maturation Line")
scatter = ax.scatter(df_hvg['Fetal_Spearman'], df_hvg['Adult_Spearman'],
                     color='darkgrey', s=100, edgecolor='k', zorder=3, label="Organoid Datasets")
ax.scatter([1.0], [baseline_ref_corr], marker='*', s=400, color='#3498db', edgecolor='k', zorder=4, label="Fetal Reference")
ax.scatter([baseline_ref_corr], [1.0], marker='*', s=400, color='#e74c3c', edgecolor='k', zorder=4, label="Adult Reference")
ax.set_title("Organoid Maturation Landscape (Unbiased HVG Pseudobulk Correlation)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel(r"Spearman Correlation ($\rho$) to Fetal Reference", fontsize=12)
ax.set_ylabel(r"Spearman Correlation ($\rho$) to Adult Reference", fontsize=12)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, shadow=True)
label_offsets = {
    "LMRH": (0.035, 0.045)              # Increased dx and dy for a longer arrow
}
for _, row in df_hvg.iterrows():
    for key, label in DATASET_LABELS.items():
        if key in row["Dataset"]:
            dx, dy = label_offsets[label]
            ax.annotate(
                label,
                xy=(row['Fetal_Spearman'], row['Adult_Spearman']),
                xytext=(row['Fetal_Spearman'] + dx,
                        row['Adult_Spearman'] + dy),
                fontsize=9,
                ha='left' if dx > 0 else 'right',
                va='bottom' if dy > 0 else 'top',
                arrowprops=dict(
                    arrowstyle='->',
                    color='gray',
                    lw=0.8,
                    shrinkA=8,
                    shrinkB=4
                )
            )
            break
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()
